# African Countries Climate Comparison Analysis
Comprehensive climate analysis comparing Ethiopia, Kenya, Sudan, Tanzania, and Nigeria (2015-2026)

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")

In [ ]:
# Data Loading - All 5 cleaned CSVs
print("Data Loading:")
print("=" * 40)

countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
all_data = []

for country in countries:
    file_path = f'../data/{country}_clean.csv'
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"Loaded {country.title()}: {df.shape}")
        all_data.append(df)
    else:
        print(f"Warning: {file_path} not found")

# Concatenate all data
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"\nCombined dataset shape: {combined_df.shape}")
    print(f"Countries included: {combined_df['Country'].unique()}")
    print(f"Date range: {combined_df['DATE'].min()} to {combined_df['DATE'].max()}")
else:
    print("No data loaded!")
    combined_df = pd.DataFrame()

# Convert DATE to datetime if needed
if 'DATE' in combined_df.columns:
    combined_df['DATE'] = pd.to_datetime(combined_df['DATE'])

combined_df.head()

In [ ]:
# Temperature Comparison
print("Temperature Comparison:")
print("=" * 40)

# Calculate yearly average temperatures
combined_df['YEAR'] = pd.to_datetime(combined_df['DATE']).dt.year
yearly_temp = combined_df.groupby(['Country', 'YEAR'])['T2M'].mean().reset_index()

# Multi-line chart (2015-2026)
plt.figure(figsize=(14, 8))
for country in yearly_temp['Country'].unique():
    country_data = yearly_temp[yearly_temp['Country'] == country]
    plt.plot(country_data['YEAR'], country_data['T2M'], 
             marker='o', linewidth=2, markersize=6, label=country)

plt.title('African Countries: Annual Average Temperature Comparison (2015-2026)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(range(2015, 2027))
plt.tight_layout()
plt.show()

# Summary table
temp_summary = combined_df.groupby('Country')['T2M'].agg([
    'mean', 'std', 'min', 'max'
]).round(2)
temp_summary.columns = ['Mean Temp (°C)', 'Std Dev', 'Min Temp (°C)', 'Max Temp (°C)']
temp_summary = temp_summary.sort_values('Mean Temp (°C)', ascending=False)

print("\nTemperature Summary Table:")
print(temp_summary)

In [ ]:
# Precipitation Comparison
print("Precipitation Comparison:")
print("=" * 40)

# Side-by-side boxplots
plt.figure(figsize=(14, 8))
sns.boxplot(data=combined_df, x='Country', y='PRECTOTCORR', palette='Blues')
plt.title('African Countries: Precipitation Distribution Comparison', 
          fontsize=14, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Precipitation (mm)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Summary table
precip_summary = combined_df.groupby('Country')['PRECTOTCORR'].agg([
    'mean', 'std', 'min', 'max', 'sum'
]).round(2)
precip_summary.columns = ['Mean Precip (mm)', 'Std Dev', 'Min Precip (mm)', 
                         'Max Precip (mm)', 'Total Precip (mm)']
precip_summary = precip_summary.sort_values('Mean Precip (mm)', ascending=False)

print("\nPrecipitation Summary Table:")
print(precip_summary)

In [ ]:
# Extreme Heat Analysis
print("Extreme Heat Analysis:")
print("=" * 40)

# Calculate days per year with T2M_MAX > 35°C
extreme_heat_days = combined_df[combined_df['T2M_MAX'] > 35].groupby(['Country', 'YEAR']).size().reset_index(name='extreme_heat_days')

# Average across years
avg_extreme_heat = extreme_heat_days.groupby('Country')['extreme_heat_days'].mean().reset_index()
avg_extreme_heat = avg_extreme_heat.sort_values('extreme_heat_days', ascending=False)

# Bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(avg_extreme_heat['Country'], avg_extreme_heat['extreme_heat_days'], 
               color='red', alpha=0.7)
plt.title('African Countries: Average Days per Year with Extreme Heat (>35°C)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Average Days per Year', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nExtreme Heat Days Summary:")
print(avg_extreme_heat)
print(f"\nMost vulnerable to extreme heat: {avg_extreme_heat.iloc[0]['Country']} "
      f"({avg_extreme_heat.iloc[0]['extreme_heat_days']:.1f} days/year)")

In [ ]:
# Dry Days Analysis
print("Dry Days Analysis:")
print("=" * 40)

# Calculate consecutive dry days (precip < 1mm)
def calculate_consecutive_dry_days(group):
    dry_days = (group['PRECTOTCORR'] < 1).astype(int)
    consecutive = []
    current_streak = 0
    
    for is_dry in dry_days:
        if is_dry:
            current_streak += 1
        else:
            if current_streak > 0:
                consecutive.append(current_streak)
            current_streak = 0
    
    if current_streak > 0:
        consecutive.append(current_streak)
    
    return max(consecutive) if consecutive else 0

# Calculate max consecutive dry days per year per country
max_dry_days = []
for country in combined_df['Country'].unique():
    country_data = combined_df[combined_df['Country'] == country]
    for year in country_data['YEAR'].unique():
        year_data = country_data[country_data['YEAR'] == year].sort_values('DATE')
        max_consecutive = calculate_consecutive_dry_days(year_data)
        max_dry_days.append({
            'Country': country,
            'YEAR': year,
            'max_consecutive_dry_days': max_consecutive
        })

dry_days_df = pd.DataFrame(max_dry_days)
avg_dry_days = dry_days_df.groupby('Country')['max_consecutive_dry_days'].mean().reset_index()
avg_dry_days = avg_dry_days.sort_values('max_consecutive_dry_days', ascending=False)

# Bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(avg_dry_days['Country'], avg_dry_days['max_consecutive_dry_days'], 
               color='orange', alpha=0.7)
plt.title('African Countries: Average Maximum Consecutive Dry Days per Year', 
          fontsize=14, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Average Maximum Consecutive Dry Days', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nConsecutive Dry Days Summary:")
print(avg_dry_days)
print(f"\nMost prone to drought: {avg_dry_days.iloc[0]['Country']} "
      f"({avg_dry_days.iloc[0]['max_consecutive_dry_days']:.1f} days)")

In [ ]:
# Statistical Test - ANOVA/Kruskal-Wallis
print("Statistical Test:")
print("=" * 40)

# Test for temperature differences between countries
temp_groups = []
for country in combined_df['Country'].unique():
    country_temps = combined_df[combined_df['Country'] == country]['T2M']
    temp_groups.append(country_temps)

# First check normality for ANOVA assumption
print("Normality tests for temperature data:")
for i, country in enumerate(combined_df['Country'].unique()):
    stat, p_value = stats.shapiro(temp_groups[i][:5000])  # Sample for performance
    print(f"{country}: W={stat:.3f}, p={p_value:.6f}")

# Since data is likely not normal, use Kruskal-Wallis test
stat, p_value = stats.kruskal(*temp_groups)

print(f"\nKruskal-Wallis Test Results:")
print(f"Statistic: {stat:.3f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("\nInterpretation: There are statistically significant differences in temperature between countries (p < 0.05)")
    
    # Post-hoc pairwise comparisons
    print("\nPairwise comparisons:")
    countries_list = list(combined_df['Country'].unique())
    for i in range(len(countries_list)):
        for j in range(i+1, len(countries_list)):
            country1, country2 = countries_list[i], countries_list[j]
            group1 = combined_df[combined_df['Country'] == country1]['T2M']
            group2 = combined_df[combined_df['Country'] == country2]['T2M']
            stat, p_val = stats.kruskal(group1, group2)
            significance = "< 0.05" if p_val < 0.05 else ">= 0.05"
            print(f"  {country1} vs {country2}: p = {p_val:.6f} {significance}")
else:
    print("\nInterpretation: No statistically significant differences in temperature between countries (p >= 0.05)")

# Effect size (eta-squared)
n_total = len(combined_df)
n_groups = len(combined_df['Country'].unique())
eta_squared = (stat - n_groups + 1) / (n_total - n_groups)
print(f"\nEffect size (η²): {eta_squared:.3f}")

if eta_squared < 0.01:
    effect_interpretation = "negligible"
elif eta_squared < 0.06:
    effect_interpretation = "small"
elif eta_squared < 0.14:
    effect_interpretation = "medium"
else:
    effect_interpretation = "large"

print(f"Effect size interpretation: {effect_interpretation}")

In [ ]:
# Vulnerability Ranking
print("Vulnerability Ranking:")
print("=" * 40)

# Calculate vulnerability metrics
vulnerability_metrics = []

for country in combined_df['Country'].unique():
    country_data = combined_df[combined_df['Country'] == country]
    
    # Temperature metrics
    avg_temp = country_data['T2M'].mean()
    max_temp = country_data['T2M_MAX'].max()
    temp_variability = country_data['T2M'].std()
    
    # Extreme heat days
    extreme_heat_days = (country_data['T2M_MAX'] > 35).sum() / len(country_data['YEAR'].unique())
    
    # Precipitation metrics
    avg_precip = country_data['PRECTOTCORR'].mean()
    precip_variability = country_data['PRECTOTCORR'].std()
    
    # Calculate max consecutive dry days
    max_dry_spells = []
    for year in country_data['YEAR'].unique():
        year_data = country_data[country_data['YEAR'] == year].sort_values('DATE')
        max_dry = calculate_consecutive_dry_days(year_data)
        max_dry_spells.append(max_dry)
    avg_max_dry_days = np.mean(max_dry_spells)
    
    # Calculate vulnerability score (higher = more vulnerable)
    # Normalize metrics (0-1 scale) and weight them
    temp_score = (avg_temp - combined_df['T2M'].min()) / (combined_df['T2M'].max() - combined_df['T2M'].min())
    heat_score = min(extreme_heat_days / 365, 1.0)  # Cap at 1
    drought_score = min(avg_max_dry_days / 365, 1.0)  # Cap at 1
    precip_var_score = precip_variability / combined_df['PRECTOTCORR'].std()
    
    # Weighted vulnerability score (weights can be adjusted)
    vulnerability_score = (temp_score * 0.3 + heat_score * 0.3 + 
                          drought_score * 0.3 + precip_var_score * 0.1)
    
    vulnerability_metrics.append({
        'Country': country,
        'Avg_Temp_C': avg_temp,
        'Max_Temp_C': max_temp,
        'Temp_Variability': temp_variability,
        'Extreme_Heat_Days_Year': extreme_heat_days,
        'Avg_Precip_mm': avg_precip,
        'Precip_Variability': precip_variability,
        'Avg_Max_Dry_Days': avg_max_dry_days,
        'Vulnerability_Score': vulnerability_score
    })

vulnerability_df = pd.DataFrame(vulnerability_metrics)
vulnerability_df = vulnerability_df.sort_values('Vulnerability_Score', ascending=False)

print("Climate Vulnerability Ranking:")
print("(Higher score = more vulnerable)")
print("=" * 50)

for idx, row in vulnerability_df.iterrows():
    rank = idx + 1
    print(f"{rank}. {row['Country']} - Score: {row['Vulnerability_Score']:.3f}")
    print(f"   Justification: High temp ({row['Avg_Temp_C']:.1f}°C), "
          f"{row['Extreme_Heat_Days_Year']:.1f} extreme heat days/year, "
          f"{row['Avg_Max_Dry_Days']:.1f} avg max dry days")
    print()

# Visual representation
plt.figure(figsize=(12, 8))
colors = ['red', 'orange', 'yellow', 'lightgreen', 'green']
bars = plt.barh(vulnerability_df['Country'], vulnerability_df['Vulnerability_Score'], 
                color=colors[:len(vulnerability_df)])
plt.title('African Countries: Climate Vulnerability Ranking', fontsize=14, fontweight='bold')
plt.xlabel('Vulnerability Score (Higher = More Vulnerable)', fontsize=12)
plt.gca().invert_yaxis()  # Highest vulnerability at top
plt.grid(True, alpha=0.3, axis='x')

# Add score labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2,
             f'{width:.3f}', ha='left', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# COP32 Bullet Points
print("COP32 Climate Action Bullet Points:")
print("=" * 50)

# Key findings for policy recommendations
cop32_points = [
    {
        'question': 'Which countries need urgent heat adaptation?',
        'answer': f"{vulnerability_df.iloc[0]['Country']} shows highest vulnerability with {vulnerability_df.iloc[0]['Extreme_Heat_Days_Year']:.1f} extreme heat days annually. Immediate heat action plans and cooling infrastructure needed."
    },
    {
        'question': 'Where should drought resilience investments focus?',
        'answer': f"{avg_dry_days.iloc[0]['Country']} experiences {avg_dry_days.iloc[0]['max_consecutive_dry_days']:.1f} consecutive dry days on average. Water conservation and drought-resistant agriculture are critical priorities."
    },
    {
        'question': 'Which regions show temperature increase trends?',
        'answer': f"Statistical analysis confirms significant temperature variations between countries (p < 0.05). {temp_summary.index[0]} has highest average temperature ({temp_summary.iloc[0, 0]:.1f}°C), indicating climate change impacts."
    },
    {
        'question': 'What precipitation patterns threaten food security?',
        'answer': f"{precip_summary.index[-1]} has lowest precipitation ({precip_summary.iloc[-1, 0]:.1f}mm average) while {precip_summary.index[0]} receives {precip_summary.iloc[0, 0]:.1f}mm. This extreme variation threatens regional food security."
    },
    {
        'question': 'How should climate finance be allocated?',
        'answer': f"Prioritize {vulnerability_df.iloc[0]['Country']} and {vulnerability_df.iloc[1]['Country']} for climate finance based on vulnerability scores. Focus on heat adaptation, water management, and agricultural resilience."
    }
]

for i, point in enumerate(cop32_points, 1):
    print(f"\n{i}. {point['question']}")
    print(f"   {point['answer']}")

print("\n" + "=" * 50)
print("KEY RECOMMENDATIONS FOR COP32:")
print("=" * 50)
print("1. Establish regional heat early warning systems")
print("2. Create cross-border water management agreements")
print("3. Fund climate-resilient agriculture research")
print("4. Develop drought insurance mechanisms")
print("5. Support renewable energy to reduce heat island effects")

In [ ]:
# Final Summary
print("AFRICAN CLIMATE TREND ANALYSIS - FINAL SUMMARY")
print("=" * 60)
print(f"Analysis Period: 2015-2026")
print(f"Countries Analyzed: {', '.join(combined_df['Country'].unique())}")
print(f"Total Data Points: {len(combined_df):,}")
print("\nKEY FINDINGS:")
print("-" * 30)
print(f"• Hottest Country: {temp_summary.index[0]} ({temp_summary.iloc[0, 0]:.1f}°C avg)")
print(f"• Wettest Country: {precip_summary.index[0]} ({precip_summary.iloc[0, 0]:.1f}mm avg precip)")
print(f"• Most Vulnerable: {vulnerability_df.iloc[0]['Country']} (Score: {vulnerability_df.iloc[0]['Vulnerability_Score']:.3f})")
print(f"• Extreme Heat Leader: {avg_extreme_heat.iloc[0]['Country']} ({avg_extreme_heat.iloc[0]['extreme_heat_days']:.1f} days/year)")
print(f"• Drought Prone: {avg_dry_days.iloc[0]['Country']} ({avg_dry_days.iloc[0]['max_consecutive_dry_days']:.1f} consecutive dry days)")
print("\nSTATISTICAL SIGNIFICANCE:")
print("-" * 30)
print(f"• Temperature differences: Statistically significant (p < 0.05)")
print(f"• Effect size: {eta_squared:.3f} ({effect_interpretation})")
print("\nPOLICY IMPLICATIONS:")
print("-" * 30)
print("• Regional climate cooperation essential")
print("• Targeted adaptation strategies needed")
print("• Climate finance should prioritize vulnerability scores")
print("• Cross-border data sharing recommended")
print("\nAnalysis completed successfully!")